# CoherenceProbe: Comprehensive Demo

**Detect logical contradictions across multi-agent AI pipeline outputs — without needing ground truth.**

## The Problem

Research shows **33-94%** of multi-agent compositions contain hidden contradictions. When you chain multiple AI agents together:

- Agent A says the server runs on port 8080
- Agent B says it runs on port 3000
- Both sound confident
- Your pipeline just told users two incompatible truths

CoherenceProbe detects these contradictions automatically.

## How It Works

**3-Stage Pipeline:**
1. **Claim Extraction** — Extract atomic factual claims from agent outputs
2. **Contradiction Detection** — Use NLI to find contradicting claim pairs
3. **Coherence Scoring** — Compute overall coherence score (0-1)

Let's explore each stage with hands-on examples!

## 1. Installation & Setup

In [ ]:
# Install CoherenceProbe (uncomment if needed)
# !pip install -e .
# !pip install -e ".[local,viz]"
# !python -m spacy download en_core_web_sm

In [ ]:
# Core imports
from coherenceprobe import (
    check, acheck,
    AgentOutput,
    LogCapture, FileCapture, DecoratorCapture,
    CoherenceConfig,
    extract_claims,
    detect_contradictions,
    compute_coherence_score,
    format_report
)

# Utilities
import json
from datetime import datetime
from pprint import pprint

print("✅ CoherenceProbe imported successfully!")

## 2. Quick Start: Simple Contradiction

In [ ]:
# Scenario: Two agents analyzing server configuration
# They give contradicting port numbers

outputs = [
    AgentOutput(
        agent="config_analyzer",
        timestamp="2026-06-07T10:00:00Z",
        input="What port does the server use?",
        output="The server runs on port 8080 and handles HTTP requests.",
        metadata={"model": "gpt-4o-mini"}
    ),
    AgentOutput(
        agent="docs_parser",
        timestamp="2026-06-07T10:00:05Z",
        input="What port does the server use?",
        output="According to the documentation, the server runs on port 3000.",
        metadata={"model": "claude-sonnet-4"}
    ),
]

# Check coherence (using local mode to avoid API calls)
config = CoherenceConfig(local=True, verbose=True)
report = check(outputs, config)

print(f"\n{'='*70}")
print(f"Coherence Score: {report.score:.2f} {'✅' if report.score > 0.8 else '⚠️' if report.score > 0.5 else '❌'}")
print(f"Contradictions Found: {len(report.contradictions)}")
print(f"Total Claims Extracted: {report.total_claims}")
print(f"Total Agents: {report.total_agents}")
print(f"{'='*70}\n")

# Show per-agent scores
print("Per-Agent Incoherence Scores:")
for agent, score in report.agent_scores.items():
    print(f"  {agent}: {score:.3f}")

# Show contradictions
if report.contradictions:
    print(f"\nDetected Contradictions:")
    for i, contradiction in enumerate(report.contradictions, 1):
        print(f"\n  [{i}] Type: {contradiction.contradiction_type.upper()}")
        print(f"      Confidence: {contradiction.confidence:.2f}")
        print(f"      {contradiction.claim_a.agent}: \"{contradiction.claim_a.text}\"")
        print(f"      {contradiction.claim_b.agent}: \"{contradiction.claim_b.text}\"")
        if contradiction.explanation:
            print(f"      Explanation: {contradiction.explanation}")

## 3. Capture Methods

CoherenceProbe provides three ways to capture agent outputs:

### 3.1 LogCapture (In-Memory)

In [ ]:
# Best for: Interactive exploration, small pipelines

capture = LogCapture()

# Simulate agent calls
capture.capture(
    agent="summarizer",
    input_data="Long article about AI safety...",
    output="The article discusses positive developments in AI safety research."
)

capture.capture(
    agent="critic",
    input_data="Long article about AI safety...",
    output="The article ignores major safety concerns and risks."
)

# Check coherence
report = check(capture.get_outputs(), CoherenceConfig(local=True))
print(f"Coherence Score: {report.score:.2f}")
print(f"Contradictions: {len(report.contradictions)}")

### 3.2 DecoratorCapture (Function Wrapping)

In [ ]:
# Best for: Wrapping existing agent functions

capture = DecoratorCapture()

@capture.agent("price_analyzer")
def analyze_price(product: str) -> str:
    # Simulate agent logic
    if product == "laptop":
        return "The laptop is priced at $999."
    return "Price not found."

@capture.agent("discount_checker")
def check_discount(product: str) -> str:
    # Simulate agent logic
    if product == "laptop":
        return "Current price is $1299 with no active discounts."
    return "No discount information."

# Run the agents
result1 = analyze_price("laptop")
result2 = check_discount("laptop")

print(f"Agent 1: {result1}")
print(f"Agent 2: {result2}")

# Check coherence
report = check(capture.get_outputs(), CoherenceConfig(local=True))
print(f"\nCoherence Score: {report.score:.2f}")

### 3.3 FileCapture (JSONL Persistence)

In [ ]:
# Best for: Long-running pipelines, debugging, archival

import os
import tempfile

# Create temporary file for demo
temp_file = tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False)
temp_path = temp_file.name
temp_file.close()

try:
    capture = FileCapture(temp_path)
    
    # Captures are automatically written to file
    capture.capture(
        agent="agent1",
        input_data="test input",
        output="test output 1"
    )
    
    capture.capture(
        agent="agent2",
        input_data="test input",
        output="test output 2"
    )
    
    # Read back from file
    print(f"File contents ({temp_path}):\n")
    with open(temp_path, 'r') as f:
        for line in f:
            print(json.dumps(json.loads(line), indent=2))
    
    # Can also load from file later
    from coherenceprobe import load_outputs_from_jsonl
    loaded_outputs = load_outputs_from_jsonl(temp_path)
    print(f"\nLoaded {len(loaded_outputs)} outputs from file")
    
finally:
    # Cleanup
    if os.path.exists(temp_path):
        os.unlink(temp_path)

## 4. Pipeline Deep Dive

Let's examine each stage of the coherence checking pipeline in detail.

### Stage 1: Claim Extraction

In [ ]:
# Create sample outputs with complex text
complex_outputs = [
    AgentOutput(
        agent="technical_writer",
        timestamp="2026-06-07T10:00:00Z",
        input="Describe the system architecture",
        output="""The system uses a microservices architecture. 
        The API gateway runs on port 8080. The database is PostgreSQL 14. 
        We deploy using Docker containers. Response time is under 200ms.""",
        metadata={}
    ),
    AgentOutput(
        agent="devops_analyst",
        timestamp="2026-06-07T10:00:10Z",
        input="Describe the system architecture",
        output="""The system is deployed as a monolith. 
        The main server listens on port 3000. We use MySQL 8.0 for data storage.
        Deployment is via Kubernetes. Average latency is 150ms.""",
        metadata={}
    )
]

# Extract claims using local mode
config = CoherenceConfig(local=True, verbose=False)
claims = extract_claims(complex_outputs, config)

print(f"Extracted {len(claims)} claims:\n")
for i, claim in enumerate(claims, 1):
    print(f"[{i}] Agent: {claim.agent}")
    print(f"    Original: {claim.text}")
    print(f"    Normalized: {claim.normalized}")
    print()

### Stage 2: Contradiction Detection

In [ ]:
# Detect contradictions between the claims
contradictions = detect_contradictions(claims, config)

print(f"Found {len(contradictions)} contradictions:\n")
for i, contradiction in enumerate(contradictions, 1):
    print(f"[{i}] {contradiction.contradiction_type.upper()} contradiction")
    print(f"    Confidence: {contradiction.confidence:.3f}")
    print(f"    Claim A ({contradiction.claim_a.agent}): {contradiction.claim_a.text}")
    print(f"    Claim B ({contradiction.claim_b.agent}): {contradiction.claim_b.text}")
    print()

### Stage 3: Coherence Scoring

In [ ]:
# Compute final coherence score
report = compute_coherence_score(claims, contradictions, config)

print("=" * 70)
print("COHERENCE REPORT")
print("=" * 70)
print(f"\nOverall Score: {report.score:.3f}")
print(f"Total Claims: {report.total_claims}")
print(f"Total Agents: {report.total_agents}")
print(f"Contradictions: {len(report.contradictions)}")

print(f"\nPer-Agent Incoherence Scores:")
for agent, score in sorted(report.agent_scores.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(score * 50)  # Simple ASCII bar chart
    print(f"  {agent:20s} {score:.3f} {bar}")

print(f"\n{'='*70}")

## 5. Contradiction Types Demo

CoherenceProbe classifies contradictions into three types.

### 5.1 Logical Contradictions

In [ ]:
logical_outputs = [
    AgentOutput(
        agent="status_checker",
        timestamp="2026-06-07T10:00:00Z",
        input="Is the system operational?",
        output="The system is currently operational and running normally.",
        metadata={}
    ),
    AgentOutput(
        agent="health_monitor",
        timestamp="2026-06-07T10:00:05Z",
        input="Is the system operational?",
        output="The system is not operational. All services are down.",
        metadata={}
    ),
]

report = check(logical_outputs, CoherenceConfig(local=True))

print("LOGICAL CONTRADICTION EXAMPLE")
print("="*70)
if report.contradictions:
    c = report.contradictions[0]
    print(f"Type: {c.contradiction_type}")
    print(f"Confidence: {c.confidence:.2f}")
    print(f"Claim A: '{c.claim_a.text}'")
    print(f"Claim B: '{c.claim_b.text}'")
    print(f"\n✅ Detected: Direct negation (A vs not-A)")
else:
    print("No contradictions detected (unexpected)")

### 5.2 Factual Contradictions

In [ ]:
factual_outputs = [
    AgentOutput(
        agent="config_reader",
        timestamp="2026-06-07T10:00:00Z",
        input="What is the server configuration?",
        output="The server is configured to listen on port 8080 with 4 worker threads.",
        metadata={}
    ),
    AgentOutput(
        agent="deployment_checker",
        timestamp="2026-06-07T10:00:05Z",
        input="What is the server configuration?",
        output="Current deployment shows the server running on port 3000 with 8 workers.",
        metadata={}
    ),
]

report = check(factual_outputs, CoherenceConfig(local=True))

print("FACTUAL CONTRADICTION EXAMPLE")
print("="*70)
if report.contradictions:
    for c in report.contradictions:
        print(f"Type: {c.contradiction_type}")
        print(f"Confidence: {c.confidence:.2f}")
        print(f"Claim A: '{c.claim_a.text}'")
        print(f"Claim B: '{c.claim_b.text}'")
        print()
    print(f"✅ Detected: Different values for the same attribute")
else:
    print("No contradictions detected")

### 5.3 Temporal Contradictions

In [ ]:
temporal_outputs = [
    AgentOutput(
        agent="timeline_analyzer",
        timestamp="2026-06-07T10:00:00Z",
        input="When did the deployment happen?",
        output="The deployment occurred before the security audit was completed.",
        metadata={}
    ),
    AgentOutput(
        agent="audit_reviewer",
        timestamp="2026-06-07T10:00:05Z",
        input="When did the deployment happen?",
        output="The deployment was done after the security audit finished.",
        metadata={}
    ),
]

report = check(temporal_outputs, CoherenceConfig(local=True))

print("TEMPORAL CONTRADICTION EXAMPLE")
print("="*70)
if report.contradictions:
    c = report.contradictions[0]
    print(f"Type: {c.contradiction_type}")
    print(f"Confidence: {c.confidence:.2f}")
    print(f"Claim A: '{c.claim_a.text}'")
    print(f"Claim B: '{c.claim_b.text}'")
    print(f"\n✅ Detected: Incompatible temporal ordering")
else:
    print("No contradictions detected")

## 6. Real-World Use Case: Multi-Agent Code Review

Simulate a code review pipeline with three specialized agents.

In [ ]:
# Sample code being reviewed
code_sample = """
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    return result
"""

# Three agents review the code
code_review_outputs = [
    AgentOutput(
        agent="security_agent",
        timestamp="2026-06-07T10:00:00Z",
        input=code_sample,
        output="""Security Review: CRITICAL ISSUE FOUND
        Line 2 contains a SQL injection vulnerability due to string interpolation.
        The user_id parameter is directly inserted into the query without sanitization.
        Recommendation: Use parameterized queries.""",
        metadata={"severity": "critical"}
    ),
    AgentOutput(
        agent="code_quality_agent",
        timestamp="2026-06-07T10:00:03Z",
        input=code_sample,
        output="""Code Quality Review:
        The function lacks error handling for database connection failures.
        Variable naming is acceptable. Function structure is simple and clear.
        The SQL query construction is standard but could use better formatting.""",
        metadata={"severity": "medium"}
    ),
    AgentOutput(
        agent="performance_agent",
        timestamp="2026-06-07T10:00:06Z",
        input=code_sample,
        output="""Performance Review:
        The query uses SELECT * which retrieves unnecessary columns.
        No caching mechanism is implemented for frequently accessed users.
        Database connection is not pooled, which may cause performance issues.
        No indexes are explicitly used in the query.""",
        metadata={"severity": "low"}
    ),
]

print("CODE REVIEW PIPELINE ANALYSIS")
print("="*70)
print(f"\nCode under review:")
print(code_sample)
print("\n" + "="*70)

# Check coherence
config = CoherenceConfig(local=True, verbose=False)
report = check(code_review_outputs, config)

print(f"\nCoherence Score: {report.score:.2f}")
print(f"Contradictions Found: {len(report.contradictions)}")

if report.contradictions:
    print(f"\n⚠️  Agents are contradicting each other:")
    for i, c in enumerate(report.contradictions, 1):
        print(f"\n[{i}] {c.claim_a.agent} vs {c.claim_b.agent}")
        print(f"    Type: {c.contradiction_type}")
        print(f"    Confidence: {c.confidence:.2f}")
        print(f"    - {c.claim_a.agent}: \"{c.claim_a.text}\"")
        print(f"    - {c.claim_b.agent}: \"{c.claim_b.text}\"")
else:
    print(f"\n✅ All agents agree! The reviews are internally consistent.")
    print(f"   Security agent found SQL injection")
    print(f"   Quality agent focused on error handling")
    print(f"   Performance agent identified optimization opportunities")
    print(f"   No contradictory claims detected.")

## 7. Configuration Options

Explore different configuration settings.

In [ ]:
# Sample outputs for testing different configs
test_outputs = [
    AgentOutput(
        agent="agent_a",
        timestamp="2026-06-07T10:00:00Z",
        input="test",
        output="The temperature is 72 degrees Fahrenheit.",
        metadata={}
    ),
    AgentOutput(
        agent="agent_b",
        timestamp="2026-06-07T10:00:01Z",
        input="test",
        output="The temperature is 22 degrees Celsius.",  # Actually the same!
        metadata={}
    ),
]

print("CONFIGURATION COMPARISON")
print("="*70)

# Test different threshold values
thresholds = [0.5, 0.7, 0.9]
print("\nTesting different NLI thresholds:")
print("(Higher threshold = more conservative, fewer contradictions flagged)\n")

for threshold in thresholds:
    config = CoherenceConfig(local=True, threshold=threshold, verbose=False)
    report = check(test_outputs, config)
    print(f"Threshold {threshold:.1f}: Score={report.score:.3f}, Contradictions={len(report.contradictions)}")

### Local vs LLM Mode

In [ ]:
print("\nMODE COMPARISON")
print("="*70)

sample_outputs = [
    AgentOutput(
        agent="agent1",
        timestamp="2026-06-07T10:00:00Z",
        input="test",
        output="""The system architecture uses microservices. 
        Each service is independently deployable. 
        The API gateway handles routing.""",
        metadata={}
    ),
]

# Local mode (spaCy)
print("\n📍 LOCAL MODE (spaCy-based extraction)")
print("   Pros: No API calls, fast, privacy-preserving")
print("   Cons: Less sophisticated claim extraction\n")

local_config = CoherenceConfig(local=True, verbose=False)
local_claims = extract_claims(sample_outputs, local_config)
print(f"   Extracted {len(local_claims)} claims:")
for i, claim in enumerate(local_claims[:5], 1):  # Show first 5
    print(f"   [{i}] {claim.text}")

# LLM mode (requires API key)
print("\n🌐 LLM MODE (AI-powered extraction)")
print("   Pros: More accurate, better claim atomization")
print("   Cons: Requires API key, costs money, latency")
print("   Note: Skipped in this demo (requires API key)")
print("   To use: config = CoherenceConfig(local=False, model='openai/gpt-4o-mini')")

## 8. Report Formats

CoherenceProbe supports multiple output formats.

In [ ]:
# Generate a sample report
sample_outputs = [
    AgentOutput(
        agent="agent_alpha",
        timestamp="2026-06-07T10:00:00Z",
        input="What is the status?",
        output="The system is online and processing requests.",
        metadata={}
    ),
    AgentOutput(
        agent="agent_beta",
        timestamp="2026-06-07T10:00:05Z",
        input="What is the status?",
        output="The system is offline for maintenance.",
        metadata={}
    ),
]

report = check(sample_outputs, CoherenceConfig(local=True))

### Text Format

In [ ]:
text_output = format_report(report, format="text")
print(text_output)

### JSON Format

In [ ]:
json_output = format_report(report, format="json")
print("JSON Output (formatted):")
print(json.dumps(json.loads(json_output), indent=2)[:1000] + "...\n[truncated]")

### HTML Format

In [ ]:
html_output = format_report(report, format="html")
print(f"HTML report generated ({len(html_output)} characters)")
print("\nPreview (first 500 chars):")
print(html_output[:500] + "...\n[truncated]")

# Optional: Save and display in notebook
# from IPython.display import HTML
# display(HTML(html_output))

## 9. Async Usage

For async workflows, use `acheck()`.

In [ ]:
import asyncio

async def async_coherence_check():
    """Demonstrate async coherence checking."""
    
    outputs = [
        AgentOutput(
            agent="async_agent_1",
            timestamp="2026-06-07T10:00:00Z",
            input="test",
            output="The API endpoint returns JSON data.",
            metadata={}
        ),
        AgentOutput(
            agent="async_agent_2",
            timestamp="2026-06-07T10:00:01Z",
            input="test",
            output="The API endpoint returns XML data.",
            metadata={}
        ),
    ]
    
    config = CoherenceConfig(local=True)
    report = await acheck(outputs, config)
    
    return report

# Run async function
print("Running async coherence check...")
report = await async_coherence_check()
print(f"✅ Async check complete!")
print(f"   Score: {report.score:.2f}")
print(f"   Contradictions: {len(report.contradictions)}")

## 10. Best Practices & Tips

### When to Use Local vs LLM Mode

**Use Local Mode (`local=True`) when:**
- Working with sensitive/confidential data
- Running in air-gapped environments
- Cost is a concern (no API fees)
- High throughput needed (no API rate limits)
- Simple text with clear factual claims

**Use LLM Mode (`local=False`) when:**
- Accuracy is critical
- Complex, nuanced text
- Need sophisticated claim extraction
- Can afford API costs
- Lower volume / batch processing

### Choosing Threshold Values

**Threshold = Confidence level for flagging contradictions**

- **0.5-0.6**: Aggressive (catches more, higher false positives)
- **0.7** (default): Balanced
- **0.8-0.9**: Conservative (fewer flags, higher precision)

**Recommendation**: Start with 0.7, tune based on your domain.

### Performance Considerations

**Complexity**: O(n²) within semantic clusters

**For large pipelines:**
1. Sample outputs rather than checking everything
2. Run on representative test cases
3. Use async mode for parallelization
4. Cache NLI model in memory for batch processing

**Typical performance:**
- 3 agents × 5 claims = ~75 NLI checks (~2-5 seconds local mode)
- 10 agents × 10 claims = ~500 NLI checks (~30-60 seconds local mode)

### Integration with CI/CD

In [ ]:
# Example: Pytest integration
print("""
# test_agent_coherence.py

import pytest
from coherenceprobe import check, CoherenceConfig

def test_pipeline_coherence(agent_outputs):
    """Ensure multi-agent pipeline produces coherent outputs."""
    
    config = CoherenceConfig(local=True, threshold=0.7)
    report = check(agent_outputs, config)
    
    # Assert high coherence
    assert report.score >= 0.8, (
        f"Coherence too low: {report.score:.2f}. "
        f"Found {len(report.contradictions)} contradictions."
    )
    
    # No critical contradictions
    critical = [c for c in report.contradictions if c.confidence >= 0.9]
    assert len(critical) == 0, f"Found {len(critical)} high-confidence contradictions"

# Run with: pytest test_agent_coherence.py
""")

## 11. Summary

### What We Covered

✅ **Quick Start**: Simple contradiction detection  
✅ **Capture Methods**: LogCapture, FileCapture, DecoratorCapture  
✅ **Pipeline Stages**: Extraction → Detection → Scoring  
✅ **Contradiction Types**: Logical, Factual, Temporal  
✅ **Real-World Use Case**: Multi-agent code review  
✅ **Configuration**: Thresholds, local vs LLM mode  
✅ **Report Formats**: Text, JSON, HTML  
✅ **Async Support**: `acheck()` for concurrent workflows  
✅ **Best Practices**: Performance, CI/CD integration  

### Key Takeaways

1. **No ground truth needed** — CoherenceProbe checks consistency, not correctness
2. **Three-stage pipeline** — Extract claims, detect contradictions, compute score
3. **Flexible deployment** — Local mode (privacy) or LLM mode (accuracy)
4. **Multiple capture methods** — Choose what fits your workflow
5. **Production-ready** — CLI, pytest integration, async support

### Next Steps

- Try CoherenceProbe on your own multi-agent pipeline
- Experiment with different configurations
- Integrate into your testing workflow
- Check out the [GitHub repo](https://github.com/yourusername/coherenceprobe)

### Resources

- 📖 **Documentation**: [README.md](../README.md)
- 📝 **Blog Post**: [BLOG.md](../BLOG.md)
- 🐛 **Issues**: [GitHub Issues](https://github.com/yourusername/coherenceprobe/issues)
- 📦 **PyPI**: [https://pypi.org/project/coherenceprobe/](https://pypi.org/project/coherenceprobe/)

---

**Built with ❤️ for making multi-agent AI systems more reliable**